# Spectrogram-Based Vishing Detector

## Imports and Configurations

In [1]:
from pathlib import Path
import sys

import torch
from torch import nn
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from modules.dataset_processing import (
    WaveformDataset,
    extract_archives,
    prepare_all_split_dataframes,
    subsample_by_class,
 )
from modules.attacks import FGSMAttack, FGSMAttackConfig
from modules.audio_processing import LogMelSpectrogramConfig, SpecAugmentConfig
from modules.models import SpectrogramCNNConfig, SpectrogramClassifier

/Users/dundale/Downloads/github/Vishing-Detection-Adversarial-Model/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_ROOT = PROJECT_ROOT / 'ASVspoof5'
MODEL_DIR = PROJECT_ROOT / 'models'
CHECKPOINT_DIR = MODEL_DIR / 'spectrogram'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

TEST_MODE = True
BONAFIDE_SUBSAMPLE = 15000
SPOOF_SUBSAMPLE = 15000
TARGET_NUM_SAMPLES = 64000
BATCH_SIZE = 64
NUM_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 10
MIN_DELTA = 1e-4
LEARNING_RATE = 1e-3
RANDOM_SEED = 67
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(RANDOM_SEED)
print({'device': str(DEVICE), 'data_root': str(DATA_ROOT), 'checkpoint_dir': str(CHECKPOINT_DIR)})

{'device': 'cpu', 'data_root': '/Users/dundale/Downloads/github/Vishing-Detection-Adversarial-Model/ASVspoof5', 'checkpoint_dir': '/Users/dundale/Downloads/github/Vishing-Detection-Adversarial-Model/models/spectrogram'}


## Data Preprocessing

In [3]:
extract_archives(DATA_ROOT, 40000, balanced=False)
split_frames = prepare_all_split_dataframes(DATA_ROOT, require_existing_files=True)

if TEST_MODE:
    split_frames['train'] = subsample_by_class(split_frames['train'], BONAFIDE_SUBSAMPLE, SPOOF_SUBSAMPLE, random_seed=RANDOM_SEED, balanced=False)
    split_frames['dev'] = subsample_by_class(split_frames['dev'], BONAFIDE_SUBSAMPLE, SPOOF_SUBSAMPLE, random_seed=RANDOM_SEED, balanced=True)
    split_frames['eval'] = subsample_by_class(split_frames['eval'], BONAFIDE_SUBSAMPLE, SPOOF_SUBSAMPLE, random_seed=RANDOM_SEED, balanced=True)

for split_name, frame in split_frames.items():
    print(split_name, frame.shape, frame['LABEL'].value_counts().to_dict())

/Users/dundale/Downloads/github/Vishing-Detection-Adversarial-Model/ASVspoof5/flac_T
/Users/dundale/Downloads/github/Vishing-Detection-Adversarial-Model/ASVspoof5/flac_T already exists.
/Users/dundale/Downloads/github/Vishing-Detection-Adversarial-Model/ASVspoof5/flac_D
/Users/dundale/Downloads/github/Vishing-Detection-Adversarial-Model/ASVspoof5/flac_D already exists.
/Users/dundale/Downloads/github/Vishing-Detection-Adversarial-Model/ASVspoof5/flac_E_eval
/Users/dundale/Downloads/github/Vishing-Detection-Adversarial-Model/ASVspoof5/flac_E_eval already exists.
train (18801, 13) {1: 15000, 0: 3801}
dev (18196, 13) {1: 9098, 0: 9098}
eval (16676, 13) {0: 8338, 1: 8338}


In [4]:
train_dataset = WaveformDataset(
    split_frames['train'],
    sample_rate=16000,
    target_num_samples=TARGET_NUM_SAMPLES,
    random_crop=True,
)
dev_dataset = WaveformDataset(
    split_frames['dev'],
    sample_rate=16000,
    target_num_samples=TARGET_NUM_SAMPLES,
    random_crop=False,
)
eval_dataset = WaveformDataset(
    split_frames['eval'],
    sample_rate=16000,
    target_num_samples=TARGET_NUM_SAMPLES,
    random_crop=False,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
eval_loader = DataLoader(eval_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

## Training 

In [5]:
frontend_config = LogMelSpectrogramConfig(sample_rate=16000, n_fft=1024, hop_length=256, n_mels=64, f_min=20.0, f_max=7600.0)
model_config = SpectrogramCNNConfig(num_classes=2, base_channels=32, dropout=0.3, embedding_dim=128)
spec_augment_config = SpecAugmentConfig(freq_mask_param=10, time_mask_param=20, num_freq_masks=2, num_time_masks=2)
model = SpectrogramClassifier(transformer_config=frontend_config, model_config=model_config, spec_augment_config=spec_augment_config).to(DEVICE)
optimizer = Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

train_label_counts = split_frames['train']['LABEL'].value_counts().sort_index()
class_weights = (len(split_frames['train']) / (len(train_label_counts) * train_label_counts)).values
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
print(f"Class weights: {class_weights}")

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
model

Class weights: tensor([2.4732, 0.6267])


SpectrogramClassifier(
  (transformer): WaveformToLogMelSpectrogram(
    (mel_spectrogram): MelSpectrogram(
      (spectrogram): Spectrogram()
      (mel_scale): MelScale()
    )
  )
  (classifier): SpectrogramCNN(
    (feature_extractor): Sequential(
      (0): ConvBlock(
        (layers): Sequential(
          (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
          (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (5): ReLU(inplace=True)
          (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
        )
      )
      (1): ConvBlock(
        (layers): Sequential(
          (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)


In [6]:
def run_epoch(model, dataloader, optimizer=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    total_examples = 0

    for batch in dataloader:
        waveforms = batch['waveform'].to(DEVICE)
        labels = batch['label'].to(DEVICE)

        if training:
            optimizer.zero_grad(set_to_none=True)

        logits = model(waveforms)
        loss = criterion(logits, labels)

        if training:
            loss.backward()
            optimizer.step()

        batch_size = labels.shape[0]
        total_examples += batch_size
        total_loss += loss.item() * batch_size

    return total_loss / max(total_examples, 1)

best_dev_loss = float('inf')
best_checkpoint_path = CHECKPOINT_DIR / 'best.pt'
latest_checkpoint_path = CHECKPOINT_DIR / 'latest.pt'
epochs_without_improvement = 0

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = run_epoch(model, train_loader, optimizer=optimizer)
    dev_loss = run_epoch(model, dev_loader, optimizer=None)

    scheduler.step(dev_loss)
    current_lr = optimizer.param_groups[0]['lr']

    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'train_loss': train_loss,
        'dev_loss': dev_loss,
    }
    epoch_checkpoint_path = CHECKPOINT_DIR / f'epoch_{epoch:02d}.pt'
    torch.save(checkpoint, epoch_checkpoint_path)
    torch.save(checkpoint, latest_checkpoint_path)

    improved = dev_loss < (best_dev_loss - MIN_DELTA)
    if improved:
        best_dev_loss = dev_loss
        epochs_without_improvement = 0
        torch.save(checkpoint, best_checkpoint_path)
    else:
        epochs_without_improvement += 1

    print({
        'epoch': epoch,
        'train_loss': round(train_loss, 5),
        'dev_loss': round(dev_loss, 5),
        'lr': current_lr,
        'saved': str(epoch_checkpoint_path),
        'best_saved': improved,
        'epochs_without_improvement': epochs_without_improvement,
    })

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"Early stopping at epoch {epoch} (best_dev_loss={best_dev_loss:.5f})")
        break

{'epoch': 1, 'train_loss': 0.40651, 'dev_loss': 1.00843, 'lr': 0.001, 'saved': '/Users/dundale/Downloads/github/Vishing-Detection-Adversarial-Model/models/spectrogram/epoch_01.pt', 'best_saved': True, 'epochs_without_improvement': 0}
{'epoch': 2, 'train_loss': 0.36555, 'dev_loss': 0.89983, 'lr': 0.001, 'saved': '/Users/dundale/Downloads/github/Vishing-Detection-Adversarial-Model/models/spectrogram/epoch_02.pt', 'best_saved': True, 'epochs_without_improvement': 0}
{'epoch': 3, 'train_loss': 0.35191, 'dev_loss': 0.71692, 'lr': 0.001, 'saved': '/Users/dundale/Downloads/github/Vishing-Detection-Adversarial-Model/models/spectrogram/epoch_03.pt', 'best_saved': True, 'epochs_without_improvement': 0}
{'epoch': 4, 'train_loss': 0.34601, 'dev_loss': 1.06821, 'lr': 0.001, 'saved': '/Users/dundale/Downloads/github/Vishing-Detection-Adversarial-Model/models/spectrogram/epoch_04.pt', 'best_saved': False, 'epochs_without_improvement': 1}
{'epoch': 5, 'train_loss': 0.34034, 'dev_loss': 0.51999, 'lr': 

## Evaluation

In [8]:
fgsm_attack = FGSMAttack(
    config=FGSMAttackConfig(
        epsilon=0.01,
        clamp_min=-1.0,
        clamp_max=1.0,
        targeted=False,
    )
)

all_labels = []
all_clean_predictions = []
all_adversarial_predictions = []
all_clean_true_probabilities = []
all_adversarial_true_probabilities = []

model.eval()
for batch in eval_loader:
    waveforms = batch['waveform'].to(DEVICE)
    labels = batch['label'].to(DEVICE)

    # Obtaining clean predictions
    with torch.no_grad():
        clean_logits = model(waveforms)
        clean_probabilities = torch.softmax(clean_logits, dim=1)
        clean_predictions = clean_probabilities.argmax(dim=1)
        clean_true_probabilities = clean_probabilities.gather(1, labels.unsqueeze(1)).squeeze(1)

    # Obtaining adversarial predictions
    attack_result = fgsm_attack.generate(model=model, inputs=waveforms, labels=labels)
    adversarial_waveforms = attack_result.adversarial_inputs

    with torch.no_grad():
        adversarial_logits = model(adversarial_waveforms)
        adversarial_probabilities = torch.softmax(adversarial_logits, dim=1)
        adversarial_predictions = adversarial_probabilities.argmax(dim=1)
        adversarial_true_probabilities = adversarial_probabilities.gather(1, labels.unsqueeze(1)).squeeze(1)

    all_labels.append(labels.detach().cpu())
    all_clean_predictions.append(clean_predictions.detach().cpu())
    all_adversarial_predictions.append(adversarial_predictions.detach().cpu())
    all_clean_true_probabilities.append(clean_true_probabilities.detach().cpu())
    all_adversarial_true_probabilities.append(adversarial_true_probabilities.detach().cpu())

all_labels = torch.cat(all_labels)
all_clean_predictions = torch.cat(all_clean_predictions)
all_adversarial_predictions = torch.cat(all_adversarial_predictions)
all_clean_true_probabilities = torch.cat(all_clean_true_probabilities)
all_adversarial_true_probabilities = torch.cat(all_adversarial_true_probabilities)

print({'num_samples': int(all_labels.shape[0])})

{'num_samples': 16676}


### Accuracy

In [ ]:
clean_accuracy = (all_clean_predictions == all_labels).float().mean().item()
adversarial_accuracy = (all_adversarial_predictions == all_labels).float().mean().item()

print(f"Clean Accuracy: {clean_accuracy*100:.4f}%")
print(f"Adversarial Accuracy: {adversarial_accuracy*100:.4f}%")

Clean Accuracy: 0.5700
Adversarial Accuracy: 0.4995


### Mean Average Confidence Degradation (MSCD)

In [13]:
delta_probabilities = all_clean_true_probabilities - all_adversarial_true_probabilities
macd = delta_probabilities.mean()

print(f"MACD: {macd:.6f}")

MACD: 0.072297


### Attack Success Rate (ASR)

In [14]:
asr = (all_clean_predictions != all_adversarial_predictions).float().mean().item()

print(f"ASR: {asr*100:.4f}%")

ASR: 7.4538%
